In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import bootstrap
from tqdm.auto import tqdm

In [2]:
paths = [
r'Z:\Skola\ING\Recommenders\repo\Ex2VecExtended\results\results_baselines\bl_knn_output.csv',
r'Z:\Skola\ING\Recommenders\repo\Ex2VecExtended\results\results_baselines\bl_proxy_output.csv',
r'Z:\Skola\ING\Recommenders\repo\Ex2VecExtended\results\results_baselines\last_item_output.csv',
r'Z:\Skola\ING\Recommenders\repo\Ex2VecExtended\results\results_baselines\random_output.csv',
r'Z:\Skola\ING\Recommenders\repo\Ex2VecExtended\predictions\extendedBase\extendedBase_output.csv',
r'Z:\Skola\ING\Recommenders\repo\Ex2VecExtended\predictions\extendedAdv\extendeddouble_output.csv',
r'Z:\Skola\ING\Recommenders\repo\Ex2VecExtended\predictions\extendedMLPLoss\extendedmlploss_output.csv',
r'Z:\Skola\ING\Recommenders\repo\Ex2VecExtended\predictions\original\original_output.csv',
r'Z:\Skola\ING\Recommenders\repo\Ex2VecExtended\results\results_baselines\most_popular_past_output.csv'
]




models = ['blknn', 'blproxy', 'last', 'random', 'extendedBase', 'extendedDouble', 'extendedmlp', 'original', 'most_popular']

In [3]:



total_resamples = 10_000
resamples_per_chunk = 100
batch = 100

confidence_level = 0.95

rng = np.random.default_rng(42)

results = []

for model, path in zip(models, paths):

    df = pd.read_csv(path)
    df["interaction_number"] = (
        df.groupby(['userId', 'trackId'], sort=False).cumcount().astype(np.int64) + 1
    )
    df = df[df['interaction_number'] == 1]
    values = (df['trackId'] == df['pred_1']).astype(int).to_numpy()

    observed_recall = values.mean()

    bootstrap_result = None
    completed = 0

    pbar = tqdm(
        total=total_resamples,
        desc=f"Bootstrapping {model}",
        unit="resample",
    )

    while completed < total_resamples:
        n_this_chunk = min(resamples_per_chunk, total_resamples - completed)

        bootstrap_result = bootstrap(
            data=(values,),
            statistic=np.mean,
            n_resamples=n_this_chunk,
            confidence_level=confidence_level,
            method="percentile",
            batch=batch,
            rng=rng,
            bootstrap_result=bootstrap_result,
        )

        completed += n_this_chunk
        pbar.update(n_this_chunk)

    pbar.close()

    ci = bootstrap_result.confidence_interval

    results.append(
        {
            "model": model,
            "n_examples": len(values),
            "observed_mrr": observed_recall,
            "ci_low": ci.low,
            "ci_high": ci.high,
            "ci_width": ci.high - ci.low,
            "standard_error": bootstrap_result.standard_error,
            "confidence_level": confidence_level,
            "bootstrap_iterations": total_resamples,
        }
    )

bootstrap_results = (
    pd.DataFrame(results)
    .sort_values("observed_mrr", ascending=False)
    .reset_index(drop=True)
)

bootstrap_results

Bootstrapping blknn:   0%|          | 0/10000 [00:00<?, ?resample/s]

Bootstrapping blproxy:   0%|          | 0/10000 [00:00<?, ?resample/s]

Bootstrapping last:   0%|          | 0/10000 [00:00<?, ?resample/s]

Bootstrapping random:   0%|          | 0/10000 [00:00<?, ?resample/s]

Bootstrapping extendedBase:   0%|          | 0/10000 [00:00<?, ?resample/s]

Bootstrapping extendedDouble:   0%|          | 0/10000 [00:00<?, ?resample/s]

Bootstrapping extendedmlp:   0%|          | 0/10000 [00:00<?, ?resample/s]

Bootstrapping original:   0%|          | 0/10000 [00:00<?, ?resample/s]

Bootstrapping most_popular:   0%|          | 0/10000 [00:00<?, ?resample/s]

,model,n_examples,observed_mrr,ci_low,ci_high,ci_width,standard_error,confidence_level,bootstrap_iterations
0,extendedmlp,26418,0.004429,0.003634,0.005224,0.001590,0.000406,0.95,10000
1,blknn,26418,0.002763,0.002158,0.003445,0.001287,0.000322,0.95,10000
2,most_popular,26418,0.002385,0.001817,0.002990,0.001173,0.000302,0.95,10000
3,extendedDouble,26418,0.002195,0.001666,0.002801,0.001136,0.000289,0.95,10000
4,extendedBase,26418,0.001968,0.001476,0.002536,0.001060,0.000271,0.95,10000
5,original,26418,0.001703,0.001249,0.002195,0.000946,0.000254,0.95,10000
6,random,26418,0.000454,0.000227,0.000719,0.000492,0.000132,0.95,10000
7,blproxy,26418,0.000000,0.000000,0.000000,0.000000,0.000000,0.95,10000
8,last,26418,0.000000,0.000000,0.000000,0.000000,0.000000,0.95,10000


In [4]:



total_resamples = 10_000
resamples_per_chunk = 100
batch = 100

confidence_level = 0.95

rng = np.random.default_rng(42)

k = 5

results = []

for model, path in zip(models, paths):

    df = pd.read_csv(path)
    df["interaction_number"] = (
        df.groupby(['userId', 'trackId'], sort=False).cumcount().astype(np.int64) + 1
    )
    df = df[df['interaction_number'] == 1]

    l = [f'pred_{i+1}' for i in range(k)]


    values = df[l].eq(df['trackId'], axis=0).astype(int).max(axis=1).to_numpy()

    observed_recall = values.mean()

    bootstrap_result = None
    completed = 0

    pbar = tqdm(
        total=total_resamples,
        desc=f"Bootstrapping {model}",
        unit="resample",
    )

    while completed < total_resamples:
        n_this_chunk = min(resamples_per_chunk, total_resamples - completed)

        bootstrap_result = bootstrap(
            data=(values,),
            statistic=np.mean,
            n_resamples=n_this_chunk,
            confidence_level=confidence_level,
            method="percentile",
            batch=batch,
            rng=rng,
            bootstrap_result=bootstrap_result,
        )

        completed += n_this_chunk
        pbar.update(n_this_chunk)

    pbar.close()

    ci = bootstrap_result.confidence_interval

    results.append(
        {
            "model": model,
            "n_examples": len(values),
            "observed_mrr": observed_recall,
            "ci_low": ci.low,
            "ci_high": ci.high,
            "ci_width": ci.high - ci.low,
            "standard_error": bootstrap_result.standard_error,
            "confidence_level": confidence_level,
            "bootstrap_iterations": total_resamples,
        }
    )

bootstrap_results = (
    pd.DataFrame(results)
    .sort_values("observed_mrr", ascending=False)
    .reset_index(drop=True)
)

bootstrap_results

Bootstrapping blknn:   0%|          | 0/10000 [00:00<?, ?resample/s]

Bootstrapping blproxy:   0%|          | 0/10000 [00:00<?, ?resample/s]

Bootstrapping last:   0%|          | 0/10000 [00:00<?, ?resample/s]

Bootstrapping random:   0%|          | 0/10000 [00:00<?, ?resample/s]

Bootstrapping extendedBase:   0%|          | 0/10000 [00:00<?, ?resample/s]

Bootstrapping extendedDouble:   0%|          | 0/10000 [00:00<?, ?resample/s]

Bootstrapping extendedmlp:   0%|          | 0/10000 [00:00<?, ?resample/s]

Bootstrapping original:   0%|          | 0/10000 [00:00<?, ?resample/s]

Bootstrapping most_popular:   0%|          | 0/10000 [00:00<?, ?resample/s]

,model,n_examples,observed_mrr,ci_low,ci_high,ci_width,standard_error,confidence_level,bootstrap_iterations
0,extendedmlp,26418,0.023658,0.021841,0.025513,0.003672,0.000936,0.95,10000
1,blknn,26418,0.013476,0.012113,0.014876,0.002763,0.000708,0.95,10000
2,extendedBase,26418,0.011659,0.010372,0.012984,0.002612,0.000662,0.95,10000
3,most_popular,26418,0.011659,0.010409,0.012984,0.002575,0.000664,0.95,10000
4,extendedDouble,26418,0.010296,0.009085,0.011545,0.002460,0.000623,0.95,10000
5,original,26418,0.009198,0.008063,0.010372,0.002309,0.000589,0.95,10000
6,random,26418,0.001741,0.001249,0.002271,0.001022,0.000256,0.95,10000
7,blproxy,26418,0.000000,0.000000,0.000000,0.000000,0.000000,0.95,10000
8,last,26418,0.000000,0.000000,0.000000,0.000000,0.000000,0.95,10000


In [5]:



total_resamples = 10_000
resamples_per_chunk = 100
batch = 100

confidence_level = 0.95

rng = np.random.default_rng(42)

k = 10

results = []

for model, path in zip(models, paths):

    df = pd.read_csv(path)
    df["interaction_number"] = (
        df.groupby(['userId', 'trackId'], sort=False).cumcount().astype(np.int64) + 1
    )
    df = df[df['interaction_number'] == 1]

    l = [f'pred_{i+1}' for i in range(k)]


    values = df[l].eq(df['trackId'], axis=0).astype(int).max(axis=1).to_numpy()

    observed_recall = values.mean()

    bootstrap_result = None
    completed = 0

    pbar = tqdm(
        total=total_resamples,
        desc=f"Bootstrapping {model}",
        unit="resample",
    )

    while completed < total_resamples:
        n_this_chunk = min(resamples_per_chunk, total_resamples - completed)

        bootstrap_result = bootstrap(
            data=(values,),
            statistic=np.mean,
            n_resamples=n_this_chunk,
            confidence_level=confidence_level,
            method="percentile",
            batch=batch,
            rng=rng,
            bootstrap_result=bootstrap_result,
        )

        completed += n_this_chunk
        pbar.update(n_this_chunk)

    pbar.close()

    ci = bootstrap_result.confidence_interval

    results.append(
        {
            "model": model,
            "n_examples": len(values),
            "observed_mrr": observed_recall,
            "ci_low": ci.low,
            "ci_high": ci.high,
            "ci_width": ci.high - ci.low,
            "standard_error": bootstrap_result.standard_error,
            "confidence_level": confidence_level,
            "bootstrap_iterations": total_resamples,
        }
    )

bootstrap_results = (
    pd.DataFrame(results)
    .sort_values("observed_mrr", ascending=False)
    .reset_index(drop=True)
)

bootstrap_results

Bootstrapping blknn:   0%|          | 0/10000 [00:00<?, ?resample/s]

Bootstrapping blproxy:   0%|          | 0/10000 [00:00<?, ?resample/s]

Bootstrapping last:   0%|          | 0/10000 [00:00<?, ?resample/s]

Bootstrapping random:   0%|          | 0/10000 [00:00<?, ?resample/s]

Bootstrapping extendedBase:   0%|          | 0/10000 [00:00<?, ?resample/s]

Bootstrapping extendedDouble:   0%|          | 0/10000 [00:00<?, ?resample/s]

Bootstrapping extendedmlp:   0%|          | 0/10000 [00:00<?, ?resample/s]

Bootstrapping original:   0%|          | 0/10000 [00:00<?, ?resample/s]

Bootstrapping most_popular:   0%|          | 0/10000 [00:00<?, ?resample/s]

,model,n_examples,observed_mrr,ci_low,ci_high,ci_width,standard_error,confidence_level,bootstrap_iterations
0,extendedmlp,26418,0.048187,0.045575,0.050799,0.005224,0.001319,0.95,10000
1,blknn,26418,0.026156,0.024264,0.028087,0.003823,0.000975,0.95,10000
2,extendedBase,26418,0.025172,0.023317,0.027065,0.003747,0.000964,0.95,10000
3,extendedDouble,26418,0.021765,0.020024,0.023545,0.003520,0.000896,0.95,10000
4,most_popular,26418,0.021236,0.019531,0.022977,0.003446,0.000894,0.95,10000
5,original,26418,0.020857,0.019116,0.022598,0.003482,0.000881,0.95,10000
6,random,26418,0.003407,0.002725,0.004126,0.001401,0.000360,0.95,10000
7,blproxy,26418,0.000076,0.000000,0.000189,0.000189,0.000053,0.95,10000
8,last,26418,0.000076,0.000000,0.000189,0.000189,0.000054,0.95,10000


In [6]:



total_resamples = 10_000
resamples_per_chunk = 100
batch = 100

confidence_level = 0.95

rng = np.random.default_rng(42)

k = 25

results = []

for model, path in zip(models, paths):

    df = pd.read_csv(path)
    df["interaction_number"] = (
        df.groupby(['userId', 'trackId'], sort=False).cumcount().astype(np.int64) + 1
    )
    df = df[df['interaction_number'] == 1]

    l = [f'pred_{i+1}' for i in range(k)]


    values = df[l].eq(df['trackId'], axis=0).astype(int).max(axis=1).to_numpy()

    observed_recall = values.mean()

    bootstrap_result = None
    completed = 0

    pbar = tqdm(
        total=total_resamples,
        desc=f"Bootstrapping {model}",
        unit="resample",
    )

    while completed < total_resamples:
        n_this_chunk = min(resamples_per_chunk, total_resamples - completed)

        bootstrap_result = bootstrap(
            data=(values,),
            statistic=np.mean,
            n_resamples=n_this_chunk,
            confidence_level=confidence_level,
            method="percentile",
            batch=batch,
            rng=rng,
            bootstrap_result=bootstrap_result,
        )

        completed += n_this_chunk
        pbar.update(n_this_chunk)

    pbar.close()

    ci = bootstrap_result.confidence_interval

    results.append(
        {
            "model": model,
            "n_examples": len(values),
            "observed_mrr": observed_recall,
            "ci_low": ci.low,
            "ci_high": ci.high,
            "ci_width": ci.high - ci.low,
            "standard_error": bootstrap_result.standard_error,
            "confidence_level": confidence_level,
            "bootstrap_iterations": total_resamples,
        }
    )

bootstrap_results = (
    pd.DataFrame(results)
    .sort_values("observed_mrr", ascending=False)
    .reset_index(drop=True)
)

bootstrap_results

Bootstrapping blknn:   0%|          | 0/10000 [00:00<?, ?resample/s]

Bootstrapping blproxy:   0%|          | 0/10000 [00:00<?, ?resample/s]

Bootstrapping last:   0%|          | 0/10000 [00:00<?, ?resample/s]

Bootstrapping random:   0%|          | 0/10000 [00:00<?, ?resample/s]

Bootstrapping extendedBase:   0%|          | 0/10000 [00:00<?, ?resample/s]

Bootstrapping extendedDouble:   0%|          | 0/10000 [00:00<?, ?resample/s]

Bootstrapping extendedmlp:   0%|          | 0/10000 [00:00<?, ?resample/s]

Bootstrapping original:   0%|          | 0/10000 [00:00<?, ?resample/s]

Bootstrapping most_popular:   0%|          | 0/10000 [00:00<?, ?resample/s]

,model,n_examples,observed_mrr,ci_low,ci_high,ci_width,standard_error,confidence_level,bootstrap_iterations
0,extendedmlp,26418,0.104134,0.100462,0.107805,0.007343,0.001880,0.95,10000
1,extendedBase,26418,0.064842,0.061890,0.067757,0.005867,0.001512,0.95,10000
2,blknn,26418,0.058710,0.055871,0.061625,0.005754,0.001455,0.95,10000
3,original,26418,0.058369,0.055530,0.061170,0.005640,0.001447,0.95,10000
4,extendedDouble,26418,0.052313,0.049625,0.055000,0.005375,0.001381,0.95,10000
5,most_popular,26418,0.044591,0.042093,0.047089,0.004997,0.001285,0.95,10000
6,random,26418,0.009160,0.008025,0.010334,0.002309,0.000585,0.95,10000
7,last,26418,0.000379,0.000151,0.000644,0.000492,0.000120,0.95,10000
8,blproxy,26418,0.000265,0.000076,0.000454,0.000379,0.000099,0.95,10000


In [7]:



total_resamples = 10_000
resamples_per_chunk = 100
batch = 100

confidence_level = 0.95

rng = np.random.default_rng(42)

k = 50

results = []

for model, path in zip(models, paths):

    df = pd.read_csv(path)
    df["interaction_number"] = (
        df.groupby(['userId', 'trackId'], sort=False).cumcount().astype(np.int64) + 1
    )
    df = df[df['interaction_number'] == 1]

    l = [f'pred_{i+1}' for i in range(k)]


    values = df[l].eq(df['trackId'], axis=0).astype(int).max(axis=1).to_numpy()

    observed_recall = values.mean()

    bootstrap_result = None
    completed = 0

    pbar = tqdm(
        total=total_resamples,
        desc=f"Bootstrapping {model}",
        unit="resample",
    )

    while completed < total_resamples:
        n_this_chunk = min(resamples_per_chunk, total_resamples - completed)

        bootstrap_result = bootstrap(
            data=(values,),
            statistic=np.mean,
            n_resamples=n_this_chunk,
            confidence_level=confidence_level,
            method="percentile",
            batch=batch,
            rng=rng,
            bootstrap_result=bootstrap_result,
        )

        completed += n_this_chunk
        pbar.update(n_this_chunk)

    pbar.close()

    ci = bootstrap_result.confidence_interval

    results.append(
        {
            "model": model,
            "n_examples": len(values),
            "observed_mrr": observed_recall,
            "ci_low": ci.low,
            "ci_high": ci.high,
            "ci_width": ci.high - ci.low,
            "standard_error": bootstrap_result.standard_error,
            "confidence_level": confidence_level,
            "bootstrap_iterations": total_resamples,
        }
    )

bootstrap_results = (
    pd.DataFrame(results)
    .sort_values("observed_mrr", ascending=False)
    .reset_index(drop=True)
)

bootstrap_results

Bootstrapping blknn:   0%|          | 0/10000 [00:00<?, ?resample/s]

Bootstrapping blproxy:   0%|          | 0/10000 [00:00<?, ?resample/s]

Bootstrapping last:   0%|          | 0/10000 [00:00<?, ?resample/s]

Bootstrapping random:   0%|          | 0/10000 [00:00<?, ?resample/s]

Bootstrapping extendedBase:   0%|          | 0/10000 [00:00<?, ?resample/s]

Bootstrapping extendedDouble:   0%|          | 0/10000 [00:00<?, ?resample/s]

Bootstrapping extendedmlp:   0%|          | 0/10000 [00:00<?, ?resample/s]

Bootstrapping original:   0%|          | 0/10000 [00:00<?, ?resample/s]

Bootstrapping most_popular:   0%|          | 0/10000 [00:00<?, ?resample/s]

,model,n_examples,observed_mrr,ci_low,ci_high,ci_width,standard_error,confidence_level,bootstrap_iterations
0,extendedmlp,26418,0.173026,0.168484,0.177568,0.009085,0.002342,0.95,10000
1,extendedBase,26418,0.116890,0.113029,0.120827,0.007798,0.001980,0.95,10000
2,original,26418,0.109849,0.106064,0.113597,0.007533,0.001931,0.95,10000
3,blknn,26418,0.101938,0.098266,0.105686,0.007419,0.001870,0.95,10000
4,extendedDouble,26418,0.093421,0.089939,0.096941,0.007003,0.001796,0.95,10000
5,most_popular,26418,0.077485,0.074230,0.080816,0.006586,0.001669,0.95,10000
6,random,26418,0.017185,0.015633,0.018775,0.003142,0.000800,0.95,10000
7,last,26418,0.001628,0.001173,0.002120,0.000946,0.000245,0.95,10000
8,blproxy,26418,0.001363,0.000946,0.001817,0.000871,0.000225,0.95,10000


In [8]:
k = 10

In [9]:
l = [f'pred_{i+1}' for i in range(k)]

In [10]:
l

['pred_1',
 'pred_2',
 'pred_3',
 'pred_4',
 'pred_5',
 'pred_6',
 'pred_7',
 'pred_8',
 'pred_9',
 'pred_10']

In [11]:
l = ['pred_1', 'pred_2']

df[l].eq(df['trackId'], axis=0).astype(int).max(axis=1)


0         0
1         0
2         0
3         0
4         0
         ..
115667    0
115668    0
115675    0
115681    0
115682    0
Length: 26418, dtype: int64